<a href="https://colab.research.google.com/github/Stubberson/project-collection/blob/main/aalto-thesis/correlation_regression.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Correlation and regression analyses
This notebook is for conducting correlation and mutliple regression. Before the statistical analysis methods, descriptive statistics were drawn for the data.

In multiple regression analysis **one** dependent variable is examined against multiple independent variables. Regression analysis helps determine the strength and nature of the relationship, and can be used for prediction and forecasting. Essentially, it helps understand how changes in the independent variables are associated with changes in the dependent variable.

Notebook structure:
1. EDA
2. Correlation matrices
3. Multiple Regression
4. Bariance Inflation Factor for multicollinearity

In [ ]:
# Imports
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from matplotlib import font_manager as fm
import seaborn as sns
from scipy import stats
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import train_test_split
import statsmodels.api as sm

In [ ]:
# Read the data for the explorative linear regression
dataset = pd.read_csv("/content/full_analysis.csv")
dataset_df = pd.DataFrame(dataset)
dataset_df.head()

## 1. EDA 🔍
Checks for regression. These are not the most important now, a small sample size accentuates e.g. the outliers:
* Outliers
* Normality

In [ ]:
# --- EXPLORATORY ANALYSIS ---
DEP_VAR = ["Y_submission_rate", "Y_zoom"]
# First set
#INDEP_VARS = ['visitor_count', 'page_count', 'word_count', 'median_minutes', 'multiple_maps', 'popup_count', 'avg_elems_page', 'avg_active_overlays_page', 'kB_rel', 'avg_geo_features', 'avg_select_elems']
INDEP_VARS = ['median_minutes', 'multiple_maps', 'popup_count', 'avg_elems_page', 'kB_rel', 'avg_geo_features', 'avg_select_elems']

In [ ]:
# --- FULL ANALYSIS ---
# Check for linearity between dependent and independent variables
DEP_VAR = ["Y_submission_rate", "Y_zoom"]
INDEP_VARS = ['median_minutes', 'avg_popup_page', 'avg_elems_page', 'avg_active_overlays', 'multiple_maps', 'customization','progression','geo_point','geo_line','geo_area','i_c','m_t','p_r','up_d']

# Largest number of X
#INDEP_VARS = ['median_minutes','multiple_maps','element_variety','avg_elems_page','avg_popup_page', 'kB_rel', 'avg_geo_features', 'avg_select_elems', 'customization','progression','geo_point','geo_line','geo_area','i_c','m_t','p_r','up_d']

In [ ]:
# Create box plots for dependent variables
plt.figure(figsize=(12, 5))

for i, dep in enumerate(DEP_VAR):
    plt.subplot(1, len(DEP_VAR), i + 1)
    boxplot = sns.boxplot(data=dataset_df[dep], fliersize=2.8) # Fliersize = outlier point size
    plt.title(f"Box plot of {dep}")
    plt.ylabel("Values")

    # Identify and annotate outliers for the current dependent variable
    Q1 = dataset_df[dep].quantile(0.25)
    Q3 = dataset_df[dep].quantile(0.75)
    IQR = Q3 - Q1
    lower_bound = Q1 - 1.5 * IQR
    upper_bound = Q3 + 1.5 * IQR

    outliers = dataset_df[(dataset_df[dep] < lower_bound) | (dataset_df[dep] > upper_bound)][dep]
    for outlier in outliers:
        plt.text(0, outlier, f'{outlier:.2f}', horizontalalignment='center', verticalalignment='bottom', size='small', color='black')

plt.tight_layout()
plt.show()

### 1.1 Normality
Check whether the dependent variables are normally distributed. Not extremely important as long as there are variance. Residuals having a normal distribution is much more importnat.

In [ ]:
# Perform the Shapiro-Wilk test for normality (Shapiro-Wilk is used for small sample sizes)
for dep in DEP_VAR:
  shapiro_test = stats.shapiro(dataset_df[dep].dropna())
  print(f"Shapiro-Wilk test statistic: {shapiro_test.statistic}, p-value: {shapiro_test.pvalue}")
  # Interpret the result
  if shapiro_test.pvalue > 0.05:
      print(f"For {dep}, the data is normally distributed (fail to reject H0).\n")
  else:
      print(f"For {dep}, the data is not normally distributed (reject H0).\n")

In [ ]:
# Create histograms for dependent variables
plt.figure(figsize=(12, 5)) # Adjust figure size as needed

for i, dep in enumerate(DEP_VAR):
    plt.subplot(1, len(DEP_VAR), i + 1) # Create subplots side-by-side
    sns.histplot(data=dataset_df, x=dep, kde=False)
    plt.title(f"Histogram of '{dep}'")
    plt.xlabel(dep)
    plt.ylabel("Survey Frequency")

plt.tight_layout()
plt.show()

In [ ]:
dataset_df[DEP_VAR]['Y_zoom_log'].to_csv('zoom_log.csv')

### 1.2 Transformations: log and power
Apply log and power transformations to normalize the skewed `zoom`. Update the `DEP_VAR`s.

In [ ]:
# Apply a log transformation to 'zoom'.
# Mirror the data to make it right skewed, and then apply the log transformation.
dataset_df['Y_zoom_log'] = np.log(0 - dataset_df['Y_zoom'] + 2)

# Update with the new log transformed variables
DEP_VAR = ["Y_submission_rate", "Y_zoom_log"]

In [ ]:
# Square the variable
dataset_df['Y_zoom_pow'] = np.pow(dataset_df['Y_zoom'], 2)

# Update with the new log transformed variables
DEP_VAR = ["Y_submission_rate", "Y_zoom_pow"]

In [ ]:
# Square the variable
dataset_df['Y_zoom_sqt'] = np.sqrt(dataset_df['Y_zoom'])

# Update with the new log transformed variables
DEP_VAR = ["Y_submission_rate", "Y_zoom_sqt"]

In [ ]:
dataset_df['Y_zoom_bc'], fitted_lambda = stats.boxcox(dataset_df['Y_zoom'])
DEP_VAR = ["Y_submission_rate", "Y_zoom_bc"]

In [ ]:
# Create histograms for independent variables
num_indep_vars = len(INDEP_VARS)
num_rows = (num_indep_vars + 1) // 2 # Calculate the number of rows needed (2 plots per row)

plt.figure(figsize=(12, num_rows * 4)) # Adjust figure size based on the number of rows

for i, dep in enumerate(INDEP_VARS):
    plt.subplot(num_rows, 2, i + 1) # Create subplots with 2 columns
    sns.histplot(data=dataset_df, x=dep)
    plt.title(f"Histogram of {dep}")
    plt.xlabel(dep)
    plt.ylabel("Frequency")

plt.tight_layout()
plt.show()

## 2. Correlations 🔗
Correlation between every variable in the analysis.

In [ ]:
# Correlation matrices
corr_matrix_p = dataset_df[list(INDEP_VARS) + list(DEP_VAR)].corr(method='pearson')
corr_matrix_s = dataset_df[list(INDEP_VARS) + list(DEP_VAR)].corr(method='spearman')

# Visualize the Pearson's matrix
plt.figure(figsize=(20, 10)) # Increased figure size
sns.heatmap(corr_matrix_p, annot=True, fmt=".2f", cmap="coolwarm", square=True, cbar_kws={"shrink": .8}, vmin=-1, vmax=1)
plt.title("Pearson's product moment correlation matrix")
plt.show()

# Visualize the Spearman's matrix
plt.figure(figsize=(20, 10)) # Increased figure size
sns.heatmap(corr_matrix_s, annot=True, fmt=".2f", cmap="coolwarm", square=True, cbar_kws={"shrink": .8}, vmin=-1, vmax=1)
plt.title("Spearman's rank correlation matrix")
plt.show()

## 3. Multiple Regression ⛳
* Build an Ordinary Least Squares regression model
* Scale the independet variables
* Statistics and diagnostics

In [ ]:
def regression_model(x, y, indep_var_names):
    '''
    Builds the OLS regression model.

    Methods:
      summary(): see the results
      predict(): predict values with unseen data

    Outputs:
      model: the regression model
      x_test_const: the independent variables' values for testing with an added constant
      y_test: the true values for testing

    Parameters:
      x: the independent variables
      y: the dependent variables
      indep_var_names: the names of the independent variables

    Option for predictive modelling:
      x_train_df, x_test_df, y_train, y_test = train_test_split(x_df, y_df, test_size=None, random_state=42)
      x_train_const = sm.add_constant(x_train_df)
      x_test_const = sm.add_constant(x_test_df)

    '''
    # Create DataFrames with column names and the original index
    x_df = pd.DataFrame(x, columns=indep_var_names, index=y.index)
    y_df = pd.DataFrame(y, index=y.index)

    x_const = sm.add_constant(x_df)

    model = sm.OLS(y_df, x_const).fit(cov_type='HC3') # Heteroscedasticity-Consistent standard errors: the model is robust even if errors don't have constant variance
    return model, x_const, y_df

def qqplot(model, name=None):
    ''' Plots the Q-Q plot for a model '''
    fig = sm.qqplot(model.resid, line='s')
    plt.title(f'Q-Q plot of residuals {name}')
    # Add legend
    plt.legend(['Residuals', 'Normal distribution line'])
    plt.show(fig)

### Only standardize the continuous variables

In [ ]:
# Scale the independent variables, excluding binary variables
scaler = StandardScaler() # Z-score scaling: (value - mean) / std_dev --> mean = 0, std_dev = 1

# Identify binary and continuous variables
binary_vars = ['multiple_maps', 'customization', 'progression', 'geo_point', 'geo_line', 'geo_area', 'i_c', 'm_t', 'p_r', 'up_d']
continuous_vars = [var for var in INDEP_VARS if var not in binary_vars]

# Separate binary and continuous variables
x_binary = dataset_df[binary_vars]
x_continuous = dataset_df[continuous_vars]

# Scale only the continuous variables
x_scaled_continuous = scaler.fit_transform(x_continuous)

# Combine the scaled continuous variables and the original binary variables
x_scaled_new = np.hstack((x_scaled_continuous, x_binary.values))

# Update INDEP_VARS to reflect the order after stacking (continuous first, then binary)
INDEP_VARS_SCALED_NEW = continuous_vars + binary_vars

In [ ]:
# Build the models for each dependent variable using the new scaled data
sr_model_new, sr_x_test_new, sr_y_test_new = regression_model(x_scaled_new, dataset_df[DEP_VAR[0]], INDEP_VARS_SCALED_NEW)
z_model_new, z_x_test_new, z_y_test_new = regression_model(x_scaled_new, dataset_df[DEP_VAR[1]], INDEP_VARS_SCALED_NEW)
#sq_model_new, sq_x_test_new, sq_y_test_new = regression_model(x_scaled_new, dataset_df[DEP_VAR[2]], INDEP_VARS_SCALED_NEW)

In [ ]:
# Print the summaries and plot the Q-Q plots for the new models
print(sr_model_new.summary())
qqplot(sr_model_new, name="Submission rate")

print(z_model_new.summary())
qqplot(z_model_new, name="Zoom rate")

#print(sq_model_new.summary())
#qqplot(sq_model_new, name="OSM Proxy (New Scaling)")

## 4. Variance Inflation Factor
Caluclate VIF to check for multicollinearity between the independent variables.

* VIF = 1: No correlation with other predictors.
* 1 < VIF < 5: Moderately correlated.
* VIF > 5 or 10: Highly correlated and a cause for concern.

In [ ]:
from statsmodels.stats.outliers_influence import variance_inflation_factor
import statsmodels.api as sm

# Add a constant to the independent variables for VIF calculation
X = dataset_df[INDEP_VARS]
X = sm.add_constant(X)

# Calculate VIF for each independent variable
vif_data = pd.DataFrame()
vif_data["feature"] = X.columns
vif_data["VIF"] = [variance_inflation_factor(X.values, i)
                   for i in range(len(X.columns))]

print("Variance Inflation Factors (VIF):")
print(vif_data)

### Predictions (OPTIONAL)
For making predictions, edit the `regression_model()` function.

In [ ]:
# Use the models to create predictions for each dependent variable
sr_pred = sr_model.predict(exog=sr_x_test)
#sq_pred = sq_model.predict(exog=sq_x_test)

In [ ]:
# Plot predicted vs. true values for each regression model

plot_data = {
    "Submitted Rate": (sr_y_test, sr_pred) # ADD A COMMA
    #"Spatial Quality": (sq_y_test, sq_pred)
}

plt.figure(figsize=(18, 5))

for i, (title, (y_test, y_pred)) in enumerate(plot_data.items()):
    plt.subplot(1, 3, i + 1)
    plt.scatter(y_test, y_pred, alpha=0.5)
    plt.plot([y_test.min(), y_test.max()], [y_test.min(), y_test.max()], 'k--', lw=2) # Add diagonal line (perfect predictions)
    plt.xlabel("True Values")
    plt.ylabel("Predicted Values")
    plt.title(title)

plt.tight_layout()
plt.show()